In [ ]:
import os
import sys
import time
import socket
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse

import requests
import pandas as pd
from dotenv import load_dotenv
from stem import Signal
from stem.control import Controller

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [ ]:
# --- Configuración ---
notebook_dir = Path(os.getcwd())

# Busca .env en la carpeta actual y en la carpeta padre
for candidate in [notebook_dir / ".env", notebook_dir.parent / ".env"]:
    if candidate.exists():
        load_dotenv(dotenv_path=candidate)
        print(f".env cargado desde: {candidate}")
        break
else:
    print("Warning: .env file not found. Usando valores por defecto.")

# Si usas Tor Browser abierto: socks5://127.0.0.1:9150
# Si usas Tor como servicio:      socks5://127.0.0.1:9050
TOR_PROXY = os.getenv("TOR_PROXY", "socks5://127.0.0.1:9150")
TOR_CONTROL_PORT = int(os.getenv("TOR_CONTROL_PORT", "9151"))
TARGET_ACCOUNT = os.getenv("TARGET_ACCOUNT", "_minecogob").strip().lstrip("@")
OUTPUT_DIR = os.getenv("OUTPUT_DIR", ".")
HEADLESS = os.getenv("HEADLESS", "1") == "1"

# Máximos para evitar que el notebook se quede colgado
MAX_NITTER_ROUNDS = int(os.getenv("MAX_NITTER_ROUNDS", "2"))
MAX_EMPTY_PAGES = int(os.getenv("MAX_EMPTY_PAGES", "3"))
PAGE_LOAD_TIMEOUT = int(os.getenv("PAGE_LOAD_TIMEOUT", "45"))
WAIT_TIMEOUT = int(os.getenv("WAIT_TIMEOUT", "25"))

# Instancias Nitter. Puedes añadir/quitar aquí si alguna deja de funcionar.
NITTER_INSTANCES = [
    "https://xcancel.com",
    "https://nitter.privacyredirect.com",
    "https://nitter.tiekoetter.com",
    "https://nuku.trabun.org",
    "https://nitter.net",
    # Fallbacks que pueden funcionar mejor según el momento y la ruta Tor.
    "https://nitter.eu",
    "https://nitter.oxt.me",
]

NITTER_BASE = NITTER_INSTANCES[0]
print(f"Instancias Nitter: {NITTER_INSTANCES}")
print(f"Cuenta objetivo: @{TARGET_ACCOUNT}")
print(f"Proxy Tor: {TOR_PROXY}")

.env cargado desde: c:\Users\NADIA\TFG\.env
Cuenta objetivo: @_minecogob
Proxy Tor: socks5://127.0.0.1:9150


In [ ]:
# --- Tor Utilities ---
def parse_proxy(proxy_url):
    """Devuelve host y puerto de un proxy tipo socks5://127.0.0.1:9150."""
    parsed = urlparse(proxy_url)
    return parsed.hostname or "127.0.0.1", parsed.port or 9150


def check_tor_socket(proxy_url=TOR_PROXY, timeout=3):
    """Comprueba que hay algo escuchando en el puerto SOCKS de Tor."""
    host, port = parse_proxy(proxy_url)
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(timeout)
    try:
        sock.connect((host, port))
        print(f"OK: Tor SOCKS activo en {host}:{port}")
        return True
    except Exception as e:
        print(f"ERROR: no hay Tor escuchando en {host}:{port}: {e}")
        return False
    finally:
        sock.close()


def check_tor_requests(proxy_url=TOR_PROXY):
    """Prueba Tor con requests antes de arrancar Selenium."""
    proxies = {"http": proxy_url, "https": proxy_url}
    try:
        r = requests.get("https://check.torproject.org/api/ip", proxies=proxies, timeout=20)
        data = r.json()
        print("Prueba Tor con requests:", data)
        return bool(data.get("IsTor"))
    except Exception as e:
        print(f"WARNING: requests no pudo verificar Tor: {e}")
        return False


def renew_tor_ip():
    """Renueva la IP de Tor si el puerto de control está disponible.
    Con Tor Browser normalmente 9151 NO está habilitado, así que no es crítico.
    """
    try:
        with Controller.from_port(port=TOR_CONTROL_PORT) as controller:
            controller.authenticate()
            controller.signal(Signal.NEWNYM)
            print("Nueva IP solicitada a Tor")
            time.sleep(8)
            return True
    except Exception as e:
        print(f"No se pudo renovar IP por control port {TOR_CONTROL_PORT}: {e}")
        print("Continuo sin renovar IP. Esto es normal si usas Tor Browser.")
        return False


In [ ]:
# --- Selenium WebDriver ---
def get_driver():
    """Crea y devuelve un Chrome driver con proxy Tor."""
    if not check_tor_socket(TOR_PROXY):
        print("Abre Tor Browser o arranca el servicio Tor antes de ejecutar el scraper.")
        return None

    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-quic")
    chrome_options.add_argument("--dns-prefetch-disable")
    chrome_options.add_argument("--disable-features=AsyncDns")
    chrome_options.add_argument("--host-resolver-rules=MAP * ~NOTFOUND , EXCLUDE 127.0.0.1")
    chrome_options.add_argument(f"--proxy-server={TOR_PROXY}")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--disable-infobars")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    chrome_options.page_load_strategy = "eager"

    if HEADLESS:
        chrome_options.add_argument("--headless=new")

    print(f"Proxy configurado en Chrome: {TOR_PROXY}")

    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
        return driver
    except Exception as e:
        print(f"Error creando driver: {e}")
        return None


def test_tor_in_selenium(driver):
    """Verifica que Selenium está navegando realmente a través de Tor."""
    try:
        driver.get("https://check.torproject.org/api/ip")
        WebDriverWait(driver, WAIT_TIMEOUT).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        body = driver.find_element(By.TAG_NAME, "body").text
        print("Respuesta check.torproject.org en Selenium:")
        print(body[:500])
        return "true" in body.lower() or '"IsTor":true' in body.replace(" ", "")
    except Exception as e:
        print(f"ERROR verificando Tor en Selenium: {e}")
        return False

In [ ]:
# --- Navegación Nitter ---
def save_debug_page(driver, name="debug_nitter"):
    """Guarda captura y HTML para inspeccionar por qué falla una instancia."""
    debug_dir = Path(OUTPUT_DIR) / "debug_nitter"
    debug_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    png = debug_dir / f"{name}_{timestamp}.png"
    html = debug_dir / f"{name}_{timestamp}.html"
    try:
        driver.save_screenshot(str(png))
        html.write_text(driver.page_source, encoding="utf-8", errors="ignore")
        print(f"Debug guardado: {png} y {html}")
    except Exception as e:
        print(f"No se pudo guardar debug: {e}")


def build_nitter_url(base, date="", cursor=""):
    if date:
        return f"{base}/search?f=tweets&q=from%3A{TARGET_ACCOUNT}+until%3A{date}"
    if cursor:
        return f"{base}/{TARGET_ACCOUNT}?cursor={cursor}"
    return f"{base}/{TARGET_ACCOUNT}"


ANTIBOT_PHRASES = [
    "verifying your request",
    "making sure you're not a bot",
    "please allow up to",
    "just a moment",
    "checking your browser",
    "captcha",
    "browser will redirect",
    "site is protected",
    "x cancelled",
    "within.website",
    "antibot",
    "access denied",
    "blocked",
]


def is_antibot_page(driver):
    source = driver.page_source.lower()
    title = driver.title.lower()
    text = source + title
    return any(phrase in text for phrase in ANTIBOT_PHRASES)


def wait_for_accessible_page(driver, timeout=20):
    start = time.time()
    while time.time() - start < timeout:
        if not is_antibot_page(driver):
            return True
        time.sleep(1)
    return not is_antibot_page(driver)


def find_tweets(driver):
    selectors = [".timeline-item", ".tweet", "article", ".status", "div.status"]
    for sel in selectors:
        tweets = driver.find_elements(By.CSS_SELECTOR, sel)
        if tweets:
            return tweets
    return []


def find_nitter(driver, date="", cursor=""):
    """Navega a Nitter probando instancias alternativas. Finito: no se queda en bucle."""
    global NITTER_BASE

    if driver is None:
        print("Driver no disponible")
        return False

    for round_i in range(1, MAX_NITTER_ROUNDS + 1):
        print(f"Ronda Nitter {round_i}/{MAX_NITTER_ROUNDS}")

        for instance in NITTER_INSTANCES:
            NITTER_BASE = instance
            url = build_nitter_url(instance, date=date, cursor=cursor)
            print(f"Probando: {url}")

            try:
                driver.get(url)
                WebDriverWait(driver, WAIT_TIMEOUT).until(
                    EC.any_of(
                        EC.presence_of_element_located((By.CSS_SELECTOR, ".timeline-item")),
                        EC.presence_of_element_located((By.CSS_SELECTOR, ".tweet")),
                        EC.presence_of_element_located((By.CSS_SELECTOR, "article")),
                        EC.presence_of_element_located((By.TAG_NAME, "body")),
                    )
                )
                time.sleep(1)
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1)

                if not wait_for_accessible_page(driver, WAIT_TIMEOUT):
                    print(f"{instance} devolvió un desafío anti-bot.")
                    save_debug_page(driver, "antibot")
                    continue

                tweets = find_tweets(driver)
                source_lower = driver.page_source.lower()

                if tweets:
                    print(f"OK con {instance}: {len(tweets)} tweets encontrados")
                    return True

                if "rate limited" in source_lower or "instance has been rate limited" in source_lower:
                    print(f"{instance} está rate-limited")
                elif "protected" in source_lower or "not found" in source_lower:
                    print(f"{instance} cargó, pero la cuenta no aparece o está bloqueada")
                else:
                    print(f"{instance} cargó, pero no encontré tweets")
                    save_debug_page(driver, "sin_tweets")

            except TimeoutException as e:
                print(f"Timeout en {instance}: {str(e)[:120]}")
                save_debug_page(driver, "timeout")
            except WebDriverException as e:
                msg = str(e).replace(",", " ")[:200]
                print(f"Falla WebDriver en {instance}: {msg}")
            except Exception as e:
                print(f"Falla {instance}: {str(e)[:200]}")

            time.sleep(2)

        print("Todas las instancias fallaron en esta ronda. Intentando renovar IP una vez...")
        renew_tor_ip()

    print("No se pudo acceder a ninguna instancia Nitter con tweets.")
    return False

In [ ]:
# --- Botón siguiente página ---
def foward_button(driver):
    """Hace click en el botón 'show more' y devuelve el nuevo cursor."""
    try:
        buttons = driver.find_elements(By.CLASS_NAME, "show-more")
        if not buttons:
            print("No se encontró botón 'show-more'")
            return None

        button = buttons[-1]
        old_url = driver.current_url
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
        time.sleep(1)

        try:
            button.click()
        except Exception:
            driver.execute_script("arguments[0].click();", button)

        time.sleep(3)
        current_url = driver.current_url

        if current_url == old_url:
            print("El click no cambió la URL")
            return None

        if "cursor=" in current_url:
            cursor_value = current_url.split("cursor=")[-1].split("&")[0]
            return cursor_value or None
        return None
    except Exception as e:
        print(f"Error en botón forward: {e}")

        return None

In [ ]:
# --- Extracción de tweets ---
def load_page(driver, csv_file="gobierno_economia.csv"):
    """Extrae tweets de la página actual y los guarda en CSV."""
    tweets_data = []

    try:
        buttons = driver.find_elements(By.CSS_SELECTOR, ".show-more, a.show-more, button.show-more")
        if buttons:
            the = buttons[-1]
            actions = ActionChains(driver)
            actions.move_to_element(the).perform()
            time.sleep(1)
            try:
                the.click()
            except Exception:
                driver.execute_script("arguments[0].click();", the)
            time.sleep(2)

        WebDriverWait(driver, 20).until(
            EC.any_of(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".timeline-item")),
                EC.presence_of_element_located((By.CSS_SELECTOR, ".tweet")),
                EC.presence_of_element_located((By.CSS_SELECTOR, "article")),
            )
        )

        tweets = find_tweets(driver)
        print(f"Encontrados {len(tweets)} tweets")

    except Exception as e:
        print(f"Error cargando página: {e}")
        return ""

    iso_date = ""

    for tweet in tweets:
        try:
            element_date = tweet.find_element(By.CLASS_NAME, "tweet-date").find_element(By.TAG_NAME, "a")
            date = element_date.get_attribute("title") or ""

            try:
                hashtag_elements = tweet.find_elements(By.XPATH, ".//div[contains(@class, 'tweet-content')]//a[contains(@href, 'q=%23')]")
                hashtags = [ht.text for ht in hashtag_elements if ht.text.startswith("#")]
            except:
                hashtags = []

            username = tweet.find_element(By.CLASS_NAME, "username").text
            content = tweet.find_element(By.CLASS_NAME, "tweet-content").text

            attachments = tweet.find_elements(By.CLASS_NAME, "attachment")
            type_content = "Texto simple"
            url_media = None

            if attachments:
                if tweet.find_elements(By.CLASS_NAME, "still-image"):
                    type_content = "Imagen"
                    try:
                        url_media = tweet.find_element(By.CLASS_NAME, "still-image").get_attribute("src")
                    except:
                        url_media = None
                elif tweet.find_elements(By.CLASS_NAME, "video-container") or tweet.find_elements(By.CLASS_NAME, "media-gif"):
                    type_content = "Video/GIF"

            stats_elements = tweet.find_elements(By.CLASS_NAME, "tweet-stat")
            comments = stats_elements[0].text.strip() if len(stats_elements) > 0 else "0"
            retweets = stats_elements[1].text.strip() if len(stats_elements) > 1 else "0"
            likes    = stats_elements[2].text.strip() if len(stats_elements) > 2 else "0"
            views    = stats_elements[3].text.strip() if len(stats_elements) > 3 else "0"

            try:
                tweet_link = tweet.find_element(By.TAG_NAME, "a").get_attribute("href")
                tweet_id = tweet_link.split("/")[-1].split("#")[0]
            except:
                tweet_id = ""

            tweet_data = {
                "id": tweet_id,
                "fecha": date,
                "username": username,
                "contenido": content,
                "hashtags": ", ".join(hashtags),
                "tipo": type_content,
                "url_media": url_media,
                "comentarios": comments,
                "retweets": retweets,
                "likes": likes,
                "views": views
            }
            tweets_data.append(tweet_data)

            if date:
                try:
                    fecha_limpia = date.replace("·", "").strip()
                    objeto_fecha = datetime.strptime(fecha_limpia, "%b %d, %Y %I:%M %p UTC")
                    iso_date = objeto_fecha.strftime("%Y-%m-%d")
                except Exception as date_error:
                    print(f"Error parseando fecha: {date_error}")

        except Exception as e:
            print(f"Error extrayendo tweet: {e}")
            continue

    if tweets_data:
        df = pd.DataFrame(tweets_data)
        csv_path = os.path.join(OUTPUT_DIR, csv_file)
        df.to_csv(csv_path, mode='a', index=False, header=not os.path.exists(csv_path))
        print(f"Guardados {len(tweets_data)} tweets en {csv_path}")

    return iso_date

In [ ]:
# --- Bucle principal ---
def main():
    """Función principal del scraping."""
    continue_scraping = True
    cursor = ""
    date = "2029-01-01"
    number_pages = 0
    empty_pages = 0

    print("Inicializando driver...")
    driver = get_driver()

    if driver is None:
        print("No se pudo crear el driver. Saliendo.")
        return

    try:
        if not test_tor_in_selenium(driver):
            print("Selenium no parece estar saliendo por Tor. Revisa TOR_PROXY.")
            return

        print("Navegando a Nitter...")
        if not find_nitter(driver, cursor=cursor):
            print("No se pudo abrir Nitter. Revisa instancias o prueba sin headless: HEADLESS=0")
            return

        while continue_scraping:
            try:
                prior_date = load_page(driver)

                if not prior_date:
                    empty_pages += 1
                    print(f"No se obtuvo fecha. Intento vacío {empty_pages}/{MAX_EMPTY_PAGES}")
                    if empty_pages >= MAX_EMPTY_PAGES:
                        print("Demasiadas páginas vacías seguidas. Fin del scraping.")
                        break
                    time.sleep(5)
                    continue

                empty_pages = 0
                new_cursor = foward_button(driver)

                if new_cursor is None:
                    print("No se obtuvo cursor. Fin del scraping.")
                    break

                if datetime.fromisoformat(date) >= datetime.fromisoformat(prior_date):
                    with open("cursores.txt", "a", encoding="utf-8") as w:
                        w.write(new_cursor + "")
                    with open("fechas.txt", "a", encoding="utf-8") as w:
                        w.write(prior_date + "")
                    number_pages += 1
                    print(f"Página {number_pages} completada. Fecha: {prior_date}")
                    date = prior_date
                    cursor = new_cursor
                else:
                    print("Fecha más antigua detectada. Reabriendo búsqueda por fecha...")
                    renew_tor_ip()
                    time.sleep(2)
                    if not find_nitter(driver, date=date):
                        print("No se pudo continuar tras reabrir Nitter.")
                        break

            except Exception as e:
                print(f"Error en bucle principal: {e}")
                save_debug_page(driver, "error_bucle")
                if not find_nitter(driver, date=date):
                    break
                time.sleep(3)

    finally:
        print("Cerrando driver...")
        driver.quit()


main()


Inicializando driver...
OK: Tor SOCKS activo en 127.0.0.1:9150
Proxy configurado en Chrome: socks5://127.0.0.1:9150
Respuesta check.torproject.org en Selenium:
{"IsTor":true,"IP":"185.181.61.203"}
Navegando a Nitter...
Ronda Nitter 1/2
Probando: https://nitter.poast.org/_minecogob?cursor=
https://nitter.poast.org cargó, pero no encontré tweets
Debug guardado: debug_nitter\sin_tweets_20260609_122418.png y debug_nitter\sin_tweets_20260609_122418.html
Probando: https://nitter.privacydev.net/_minecogob?cursor=
Falla WebDriver en https://nitter.privacydev.net: Message: unknown error: net::ERR_SOCKS_CONNECTION_FAILED
  (Session info: chrome=148.0.7778.217)
Stacktrace:
	chromedriver!GetHandleVerifier [0xeab593+105d3]
	chromedriver!GetHandleVerifier [0xeab6c4+
Probando: https://nitter.1d4.us/_minecogob?cursor=
Falla WebDriver en https://nitter.1d4.us: Message: unknown error: net::ERR_SOCKS_CONNECTION_FAILED
  (Session info: chrome=148.0.7778.217)
Stacktrace:
	chromedriver!GetHandleVerifier [0x